In [2]:
from google.colab import files
uploaded = files.upload()

Saving Telco_Customer_Churn_Dataset  (1).csv to Telco_Customer_Churn_Dataset  (1).csv


In [4]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder

# Load dataset
df = pd.read_csv('Telco_Customer_Churn_Dataset  (1).csv')
print("Shape:", df.shape)
print(df.head())

# Fix TotalCharges
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'].fillna(df['TotalCharges'].median(), inplace=True)

# Drop customerID
df.drop('customerID', axis=1, inplace=True)

# Encode text columns
le = LabelEncoder()
for col in df.select_dtypes(include=['object']).columns:
    df[col] = le.fit_transform(df[col])

print("\nAfter preprocessing:", df.shape)
print(df.head())

Shape: (7043, 21)
   customerID  gender  SeniorCitizen Partner Dependents  tenure PhoneService  \
0  7590-VHVEG  Female              0     Yes         No       1           No   
1  5575-GNVDE    Male              0      No         No      34          Yes   
2  3668-QPYBK    Male              0      No         No       2          Yes   
3  7795-CFOCW    Male              0      No         No      45           No   
4  9237-HQITU  Female              0      No         No       2          Yes   

      MultipleLines InternetService OnlineSecurity  ... DeviceProtection  \
0  No phone service             DSL             No  ...               No   
1                No             DSL            Yes  ...              Yes   
2                No             DSL            Yes  ...               No   
3  No phone service             DSL            Yes  ...              Yes   
4                No     Fiber optic             No  ...               No   

  TechSupport StreamingTV StreamingMovies   

/tmp/ipykernel_4416/3862701114.py:12: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['TotalCharges'].fillna(df['TotalCharges'].median(), inplace=True)


In [5]:
# Task 2: Split Data for Training and Testing
from sklearn.model_selection import train_test_split

X = df.drop('Churn', axis=1)
y = df['Churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Training size:", len(X_train))
print("Testing size:", len(X_test))

Training size: 5634
Testing size: 1409


In [6]:
# Task 3: Feature Selection
import matplotlib.pyplot as plt
import seaborn as sns

# Find correlation of each feature with Churn
correlation = df.corr()['Churn'].sort_values(ascending=False)

print("Feature correlation with Churn:")
print(correlation)

# Select top important features
selected_features = ['tenure', 'Contract', 'TotalCharges',
                     'MonthlyCharges', 'TechSupport',
                     'OnlineSecurity', 'PaperlessBilling',
                     'InternetService']

print("\nSelected features for prediction:")
print(selected_features)

# Update X_train and X_test with selected features only
X_train_selected = X_train[selected_features]
X_test_selected = X_test[selected_features]

print("\nNew training set shape:", X_train_selected.shape)
print("New testing set shape:", X_test_selected.shape)

Feature correlation with Churn:
Churn               1.000000
MonthlyCharges      0.193356
PaperlessBilling    0.191825
SeniorCitizen       0.150889
PaymentMethod       0.107062
MultipleLines       0.038037
PhoneService        0.011942
gender             -0.008612
StreamingTV        -0.036581
StreamingMovies    -0.038492
InternetService    -0.047291
Partner            -0.150448
Dependents         -0.164221
DeviceProtection   -0.178134
OnlineBackup       -0.195525
TotalCharges       -0.199037
TechSupport        -0.282492
OnlineSecurity     -0.289309
tenure             -0.352229
Contract           -0.396713
Name: Churn, dtype: float64

Selected features for prediction:
['tenure', 'Contract', 'TotalCharges', 'MonthlyCharges', 'TechSupport', 'OnlineSecurity', 'PaperlessBilling', 'InternetService']

New training set shape: (5634, 8)
New testing set shape: (1409, 8)


In [7]:
# Task 4: Model Selection
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# Try 3 different models
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(random_state=42)
}

print("Comparing models...\n")
for name, model in models.items():
    model.fit(X_train_selected, y_train)
    y_pred = model.predict(X_test_selected)
    acc = accuracy_score(y_test, y_pred)
    print(f"{name}: Accuracy = {round(acc*100, 2)}%")

Comparing models...

Logistic Regression: Accuracy = 78.71%
Decision Tree: Accuracy = 72.89%
Random Forest: Accuracy = 77.64%


In [8]:
# Task 5: Model Training
from sklearn.ensemble import RandomForestClassifier

# Random Forest gave the best accuracy in Task 4
best_model = RandomForestClassifier(n_estimators=100, random_state=42)

# Train the model
best_model.fit(X_train_selected, y_train)

print("Model trained successfully! ✅")
print("\nModel details:")
print(f"Algorithm    : Random Forest Classifier")
print(f"Training size: {len(X_train_selected)} samples")
print(f"Features used: {selected_features}")

# Make predictions
y_pred = best_model.predict(X_test_selected)
y_prob = best_model.predict_proba(X_test_selected)[:, 1]  # needed for ROC-AUC

print(f"\nPredictions made on {len(y_pred)} test samples!")

print("\nSample predictions (first 10):")
print("Predicted:", [int(x) for x in y_pred[:10]])
print("Actual:   ", [int(x) for x in y_test[:10]])

print(f"\nOut of {len(y_pred)} customers:")
print(f"Predicted to Churn : {sum(y_pred == 1)}")
print(f"Predicted to Stay  : {sum(y_pred == 0)}")

Model trained successfully! ✅

Model details:
Algorithm    : Random Forest Classifier
Training size: 5634 samples
Features used: ['tenure', 'Contract', 'TotalCharges', 'MonthlyCharges', 'TechSupport', 'OnlineSecurity', 'PaperlessBilling', 'InternetService']

Predictions made on 1409 test samples!

Sample predictions (first 10):
Predicted: [0, 0, 0, 0, 0, 1, 0, 0, 0, 0]
Actual:    [0, 0, 0, 0, 0, 0, 0, 0, 0, 1]

Out of 1409 customers:
Predicted to Churn : 297
Predicted to Stay  : 1112


In [10]:
# Task 6: Model Evaluation
from sklearn.metrics import (accuracy_score, precision_score,
                             recall_score, f1_score,
                             roc_auc_score, classification_report)

# Get probabilities for ROC-AUC (must be before metrics)
y_prob = best_model.predict_proba(X_test_selected)[:, 1]

# Calculate all metrics
accuracy  = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall    = recall_score(y_test, y_pred)
f1        = f1_score(y_test, y_pred)
roc_auc   = roc_auc_score(y_test, y_prob)

print("=" * 40)
print("      MODEL EVALUATION RESULTS")
print("=" * 40)
print(f"Accuracy  : {round(accuracy*100, 2)}%")
print(f"Precision : {round(precision*100, 2)}%")
print(f"Recall    : {round(recall*100, 2)}%")
print(f"F1-Score  : {round(f1*100, 2)}%")
print(f"ROC-AUC   : {round(roc_auc*100, 2)}%")
print("=" * 40)

print("\nDetailed Report:")
print(classification_report(y_test, y_pred,
      target_names=['Not Churn', 'Churn']))

      MODEL EVALUATION RESULTS
Accuracy  : 77.64%
Precision : 59.93%
Recall    : 47.59%
F1-Score  : 53.06%
ROC-AUC   : 80.39%

Detailed Report:
              precision    recall  f1-score   support

   Not Churn       0.82      0.89      0.85      1035
       Churn       0.60      0.48      0.53       374

    accuracy                           0.78      1409
   macro avg       0.71      0.68      0.69      1409
weighted avg       0.76      0.78      0.77      1409

